# Etapa 1 · RAG mínimo

Objetivo: responder preguntas sobre **un solo documento**, sin base de datos.

Bloques:
1. Leer el PDF (página por página)
2. Cortarlo en chunks, guardando de qué página viene cada uno
3. Calcular los embeddings *(siguiente sesión)*
4. Buscar y responder *(siguiente sesión)*

In [ ]:
from pathlib import Path
import pypdf

## Bloque 1 · Leer el PDF

A diferencia de `read_pdf` del profe (que junta todo el texto en un solo string), aquí guardamos **cada página por separado**. Así, más adelante, el asistente podrá decir *"esto sale de la página 4"*.

In [ ]:
def leer_pdf(ruta):
    """Devuelve una lista de diccionarios: uno por página, con su número y su texto."""
    lector = pypdf.PdfReader(ruta)
    paginas = []
    for numero, pagina in enumerate(lector.pages, start=1):   # start=1 para que la primera sea la página 1, no la 0
        texto = pagina.extract_text() or ""                   # algunas páginas (p. ej. solo imagen) no tienen texto
        paginas.append({"pagina": numero, "texto": texto})
    return paginas

In [ ]:
RUTA_PDF = Path("data/rag_papers/srivastava2014_dropout.pdf")   # cambia esta ruta por tu documento

paginas = leer_pdf(RUTA_PDF)
print("Páginas:", len(paginas))
print("Caracteres totales:", sum(len(p["texto"]) for p in paginas))
print("\n--- Página 2 (primeros 500 caracteres) ---")
print(paginas[1]["texto"][:500])

## Bloque 2 · Cortar en chunks

Misma idea que `naive_chunks` del profe: ventanas de `tamano` caracteres que avanzan de `tamano - solapamiento` en `tamano - solapamiento`.

La diferencia: cortamos **página por página** y a cada chunk le pegamos su *etiqueta* (documento, página, número de chunk). Es como escribir en el reverso de cada ficha de dónde salió.

In [ ]:
def cortar_en_chunks(paginas, nombre_documento, tamano=1000, solapamiento=200):
    if not 0 <= solapamiento < tamano:
        raise ValueError("El solapamiento debe ser >= 0 y menor que el tamaño")

    paso = tamano - solapamiento
    chunks = []
    for p in paginas:
        texto = p["texto"]
        for inicio in range(0, len(texto), paso):
            trozo = texto[inicio:inicio + tamano]
            if len(trozo.strip()) < 50:          # descartamos restos casi vacíos al final de una página
                continue
            chunks.append({
                "documento": nombre_documento,
                "pagina": p["pagina"],
                "chunk_num": len(chunks),
                "texto": trozo,
            })
            if inicio + tamano >= len(texto):   # ya llegamos al final de la página
                break
    return chunks

In [ ]:
chunks = cortar_en_chunks(paginas, nombre_documento=RUTA_PDF.stem)

print("Número de chunks:", len(chunks))
ejemplo = chunks[5]
print(f"\n--- Chunk {ejemplo['chunk_num']} | {ejemplo['documento']} | página {ejemplo['pagina']} ---")
print(ejemplo["texto"])

### Para mirar con tus propios ojos

- ¿Hay chunks que empiezan con una palabra cortada por la mitad?
- ¿Las fórmulas se leen bien? (pista: busca "2N" en el paper de dropout; en el original es 2 elevado a N)
- Prueba `tamano=300` y `tamano=3000`: ¿cuántos chunks salen en cada caso?